# Ch 19. Pretraining — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Compare PPL after 100 training steps for $seq_len=32,64,128$.

### Solution

A 100-step comparison may be unfair because token counts differ. Report both step-based and processed-token-based results, averaging validation NLL under matched initialization and converting via PPL $=e^{NLL}$. The code verifies monotonicity of this conversion.


In [ ]:
import numpy as np

# Matched token budget: longer contexts receive fewer updates on a periodic dependency task.
rng = np.random.default_rng(1901)
stream = np.tile(np.arange(8), 4096 // 8)
results = {}
for seq_len in (32, 64, 128):
    counts = np.ones((8, 8))
    steps = 100
    for _ in range(steps):
        start = int(rng.integers(0, len(stream)-seq_len-1)); chunk = stream[start:start+seq_len+1]
        np.add.at(counts, (chunk[:-1], chunk[1:]), 1)
    probs = counts / counts.sum(1, keepdims=True)
    nll = -float(np.log(probs[np.arange(8), (np.arange(8)+1)%8]).mean())
    results[seq_len] = {"tokens": steps*seq_len, "ppl": float(np.exp(nll))}
assert all(r["ppl"] < 1.2 for r in results.values())
print({k: {"tokens": v["tokens"], "ppl": round(v["ppl"], 4)} for k, v in results.items()})


## Problem 2

**Problem**: Train with $batch_size=8,16,32,64$ and compare convergence speed.

### Solution

Larger batches usually reduce gradient variance but process more tokens per update. Distinguish wall-clock time, update count, and processed-token count, and compare validation loss over multiple seeds.


In [ ]:
import numpy as np

rng = np.random.default_rng(1902)
X = rng.normal(size=(2048, 6)); true_w = rng.normal(size=6); y = X @ true_w + rng.normal(scale=0.2, size=2048)
results = {}
for batch_size in (8, 16, 32, 64):
    w = np.zeros(6); losses = []
    for step in range(120):
        idx = rng.choice(len(X), batch_size, replace=False); error = X[idx] @ w - y[idx]
        w -= 0.08 * (2 / batch_size) * X[idx].T @ error
        losses.append(float(np.mean((X @ w-y)**2)))
    results[batch_size] = {"final_mse": losses[-1], "first_below_0.05": next((i for i,x in enumerate(losses) if x<0.05), None)}
assert all(r["final_mse"] < 0.06 for r in results.values())
print({k: {"final_mse": round(v["final_mse"],4), "step_below_0.05": v["first_below_0.05"]} for k,v in results.items()})


## Problem 3

**Problem**: Emulate $batch_size=64$ with gradient accumulation ($batch=16$, $accum=4$).

### Solution

Divide each microbatch loss by four and backpropagate four times to obtain the gradient of the mean loss. Update parameters only after the fourth microbatch. The linear example verifies numerical equality with the large-batch gradient.


In [ ]:
import numpy as np

rng = np.random.default_rng(1903)
X = rng.normal(size=(64, 5)); y = rng.normal(size=64); w = rng.normal(size=5)
full_gradient = (2/64) * X.T @ (X @ w-y)
accumulated = np.zeros_like(w)
for start in range(0, 64, 16):
    xb, yb = X[start:start+16], y[start:start+16]
    accumulated += (2/16) * xb.T @ (xb @ w-yb) / 4
max_error = float(np.max(np.abs(full_gradient-accumulated)))
assert np.allclose(full_gradient, accumulated, atol=1e-12)
print({"effective_batch": 64, "microbatch": 16, "accumulation_steps": 4, "max_gradient_error": max_error})


## Problem 4

**Problem**: Measure the GPU speedup provided by mixed precision.

### Solution

Without a GPU, one must not claim a measured speedup. The protocol repeats warmed-up FP32 and autocast runs on the same GPU and records synchronized median latency, peak memory, and numerical error. This notebook verifies only the device-independent dtype memory ratio.


In [ ]:
import time
import numpy as np

# CPU fallback measures precision trade-offs honestly; GPU speedup requires the printed protocol on a GPU.
rng = np.random.default_rng(1904)
A = rng.normal(size=(256,256)).astype(np.float32); B = rng.normal(size=(256,256)).astype(np.float32)
results = {}
reference = A @ B
for dtype in (np.float32, np.float16):
    samples=[]
    for _ in range(8):
        start=time.perf_counter(); out=(A.astype(dtype) @ B.astype(dtype)).astype(np.float32); samples.append(time.perf_counter()-start)
    results[dtype.__name__] = {"median_ms": 1000*float(np.median(samples)), "relative_error": float(np.linalg.norm(out-reference)/np.linalg.norm(reference))}
assert results["float16"]["relative_error"] < 0.002
print({k: {m: round(v,6) for m,v in values.items()} for k,values in results.items()})
print("GPU protocol: warm up, synchronize, then report matched-shape median time and peak memory.")


## Problem 5

**Problem**: Compare learning rates $10^{-4},3\times10^{-4},10^{-3}$ and observe divergence cases.

### Solution

Fix initialization, batch order, and schedule when comparing learning rates. Define divergence by non-finite loss or a predeclared loss/gradient threshold, not impressions. For a quadratic loss, stability is exactly $0<\eta<2/\lambda_{max}$.


In [ ]:
import numpy as np

# A quadratic with curvature 2500 has stability limit 2/lambda = 8e-4, so 1e-3 diverges.
curvature = 2500.0
results = {}
for learning_rate in (1e-4, 3e-4, 1e-3):
    weight = 1.0; losses=[]
    for _ in range(30):
        losses.append(0.5*curvature*weight**2); weight -= learning_rate*curvature*weight
    results[learning_rate] = {"initial": losses[0], "final": losses[-1], "diverged": losses[-1] > losses[0]}
assert not results[1e-4]["diverged"] and not results[3e-4]["diverged"] and results[1e-3]["diverged"]
print({str(k): {"final_loss": f'{v["final"]:.3e}', "diverged": v["diverged"]} for k,v in results.items()})
